# Compliance Copilot — Phase 2: Fine-tune the Brain

This notebook is **self-contained**: it does not read any files from the rest of
the repo. You can upload just this one file to a fresh Google Colab, run it top
to bottom, and it will rebuild the training data itself, fine-tune the model,
and push the result to the Hugging Face Hub.

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a
**T4 GPU** (free tier). Everything below assumes a GPU is available.

**What this notebook proves (for the resume line):** we measure accuracy on the
same 600 held-out contract clauses *before* fine-tuning (zero-shot) and *after*
fine-tuning (QLoRA), so the "+X% accuracy" number is real, not invented.

## Step 1 — Install the libraries

Think of this as gathering tools before starting a job: `transformers` loads the
model, `peft` + `bitsandbytes` let us load and train it in 4-bit (so it fits on a
free GPU), `trl` gives us a ready-made trainer for chat-style fine-tuning, and
`datasets` / `scikit-learn` handle the data and the accuracy scoring.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets scikit-learn


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > select a GPU (T4), then re-run.")


## Step 2 — Log in to Hugging Face

We need a Hugging Face account token for two reasons: to download the base
model, and — at the very end — to upload ("push") our trained adapter back to
the Hub so Phase 3 can load it. Running the cell below shows a login widget;
paste a token with **write** access (create one at
huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## Step 3 — Configuration

Everything you might want to change lives in one place. `BASE_MODEL` is the
small 1-billion-parameter model we fine-tune — it's an open, ungated mirror of
Llama-3.2-1B-Instruct, so it downloads without needing Meta's access-request
form. Set `HF_USERNAME` to your Hugging Face username; the trained adapter gets
uploaded to `<HF_USERNAME>/compliance-copilot-llama32-1b-lora`.

In [ ]:
import random
from collections import Counter

import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# --- Things you are likely to change ---
BASE_MODEL = "unsloth/Llama-3.2-1B-Instruct"  # ungated open copy of Llama-3.2-1B-Instruct
HF_USERNAME = "SaitejaDubbas"  # <-- your Hugging Face username
ADAPTER_REPO = f"{HF_USERNAME}/compliance-copilot-llama32-1b-lora"

# --- Data settings copied from data/prepare_data.py (must match Phase 1 exactly) ---
DATASET_ID = "coastalcph/lex_glue"
DATASET_CONFIG = "ledgar"
TOP_K_LABELS = 10
MAX_PER_LABEL_TRAIN = 400
MAX_PER_LABEL_EVAL = 60
SEED = 42

random.seed(SEED)


## Step 3b — Rebuild the dataset (same recipe as `data/prepare_data.py`)

This notebook can't read files from the repo, so we rebuild the exact same
train/val/test split here, using the identical logic, seed, and prompt wording
as `data/prepare_data.py`. That match matters: if the wording here drifted from
what Phase 3 uses to serve the model, the fine-tuned model would see unfamiliar
phrasing at inference time and get confused — like practicing for an interview
with one set of questions and then being asked completely different ones on the
day.

We download LEDGAR, keep the 10 most common clause types, and balance each
split so no single clause type dominates training.

In [ ]:
# Copied verbatim from data/prepare_data.py so training and serving never
# disagree on the wording the model was taught with.
SYSTEM_PROMPT = (
    "You are a compliance assistant. You read a single contract clause and reply "
    "with the one clause type it belongs to, and nothing else."
)


def build_user_prompt(clause_text: str, labels: list) -> str:
    label_menu = ", ".join(labels)
    return (
        f"Classify the following contract clause into exactly one of these types: "
        f"{label_menu}.\n\n"
        f"Clause:\n{clause_text}\n\n"
        f"Clause type:"
    )


print("Downloading LEDGAR ...")
ds = load_dataset(DATASET_ID, DATASET_CONFIG)

label_names = ds["train"].features["label"].names
counts = Counter(ds["train"]["label"])
top_ids = [label_id for label_id, _ in counts.most_common(TOP_K_LABELS)]
kept_labels = [label_names[i] for i in top_ids]
id_to_name = {i: label_names[i] for i in top_ids}
print(f"Keeping the top {TOP_K_LABELS} labels: {kept_labels}")


def collect(split_name: str, cap_per_label: int):
    buckets = {i: [] for i in top_ids}
    for row in ds[split_name]:
        lid = row["label"]
        if lid in buckets and len(buckets[lid]) < cap_per_label:
            buckets[lid].append(row["text"])
    pairs = [(text, id_to_name[lid]) for lid, texts in buckets.items() for text in texts]
    random.shuffle(pairs)
    return pairs


train_pairs = collect("train", MAX_PER_LABEL_TRAIN)
val_pairs = collect("validation", MAX_PER_LABEL_EVAL)
test_pairs = collect("test", MAX_PER_LABEL_EVAL)
print(f"train={len(train_pairs)}  val={len(val_pairs)}  test={len(test_pairs)}")


## Step 4 — Baseline (zero-shot) evaluation

Before we teach the model anything, let's see how well it does "cold" — just
handed the clause and the list of possible types, with no training on our
data. This is the "before" photo. We load the base model in 4-bit precision (a
compressed format that trades a little precision for using ~4x less GPU
memory, which is what makes this fit on a free T4), then ask it to classify
all 600 held-out test clauses.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched generation

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id


In [ ]:
import difflib


def parse_prediction(raw_text: str, labels: list) -> str:
    """Turn whatever text the model generated into the closest known label."""
    cleaned = raw_text.strip()
    for label in labels:
        if cleaned.lower() == label.lower():
            return label
    for label in labels:
        if label.lower() in cleaned.lower():
            return label
    close = difflib.get_close_matches(cleaned, labels, n=1, cutoff=0.0)
    return close[0] if close else labels[0]


@torch.no_grad()
def evaluate(model, tokenizer, pairs, labels, batch_size=16, max_new_tokens=8):
    model.eval()
    preds, golds = [], []
    for i in tqdm(range(0, len(pairs), batch_size)):
        batch = pairs[i : i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(text, labels)},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for text, _ in batch
        ]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024
        ).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = out[:, inputs["input_ids"].shape[1] :]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        for (text, gold), raw in zip(batch, decoded):
            preds.append(parse_prediction(raw, labels))
            golds.append(gold)
    acc = accuracy_score(golds, preds)
    f1 = f1_score(golds, preds, labels=labels, average="macro", zero_division=0)
    return acc, f1, preds, golds


In [ ]:
print("Running BASELINE (zero-shot) evaluation on the 600 held-out test clauses ...")
baseline_acc, baseline_f1, _, _ = evaluate(model, tokenizer, test_pairs, kept_labels)
print(f"\nBaseline  accuracy: {baseline_acc:.4f}   macro-F1: {baseline_f1:.4f}")


## Step 5 — Fine-tune with QLoRA

Instead of retraining all 1 billion parameters (slow, memory-hungry, and
unnecessary), QLoRA freezes the base model and trains small "adapter"
matrices bolted onto the attention layers — like adding sticky notes to a
textbook instead of rewriting it. `r=16` / `alpha=32` control how much
capacity those sticky notes have. We train for 2 passes (epochs) over the
4,000 training examples.

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
def to_chat_examples(pairs):
    return [
        {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(text, kept_labels)},
                {"role": "assistant", "content": label},
            ]
        }
        for text, label in pairs
    ]


def add_text_field(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}


train_ds = Dataset.from_list(to_chat_examples(train_pairs)).map(add_text_field)
val_ds = Dataset.from_list(to_chat_examples(val_pairs)).map(add_text_field)


In [ ]:
tokenizer.padding_side = "right"  # standard for training

sft_config = SFTConfig(
    output_dir="qlora-out",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_config,
)

trainer.train()


## Step 6 — Fine-tuned evaluation

Now we run the *exact same* evaluation function, on the *exact same* 600 test
clauses, but with the trained adapter active. This is the "after" photo —
directly comparable to Step 4 because nothing else changed.

In [ ]:
tokenizer.padding_side = "left"  # back to left-padding for generation
print("Running FINE-TUNED evaluation on the same 600 held-out test clauses ...")
finetuned_acc, finetuned_f1, _, _ = evaluate(model, tokenizer, test_pairs, kept_labels)
print(f"\nFine-tuned  accuracy: {finetuned_acc:.4f}   macro-F1: {finetuned_f1:.4f}")


## Step 7 — Results: baseline vs fine-tuned

This is the number that goes on the resume. It's measured, not invented —
both rows below ran on the identical held-out test set.

In [ ]:
print("=" * 50)
print(f"{'Metric':<12}{'Baseline':>12}{'Fine-tuned':>14}{'Improvement':>14}")
print("-" * 50)
print(
    f"{'Accuracy':<12}{baseline_acc:>12.4f}{finetuned_acc:>14.4f}"
    f"{(finetuned_acc - baseline_acc) * 100:>13.2f}pp"
)
print(
    f"{'Macro-F1':<12}{baseline_f1:>12.4f}{finetuned_f1:>14.4f}"
    f"{(finetuned_f1 - baseline_f1) * 100:>13.2f}pp"
)
print("=" * 50)


## Step 8 — Save and publish the adapter

The LoRA adapter is tiny (a few MB, since we only trained the sticky notes,
not the whole textbook), so we save it locally and push it to your Hugging
Face Hub account. Phase 3's FastAPI service will load the base model plus
this adapter by its Hub repo id — `<HF_USERNAME>/compliance-copilot-llama32-1b-lora`.

The cell below re-derives `HF_USERNAME` from your logged-in token via `whoami()` right before pushing, so a stale placeholder value in Step 3 can never cause a push to the wrong (nonexistent) namespace.

In [ ]:
from huggingface_hub import whoami

HF_USERNAME = whoami()["name"]
ADAPTER_REPO = f"{HF_USERNAME}/compliance-copilot-llama32-1b-lora"

LOCAL_ADAPTER_DIR = "compliance-copilot-lora-adapter"

model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)

print(f"Pushing adapter + tokenizer to https://huggingface.co/{ADAPTER_REPO} ...")
model.push_to_hub(ADAPTER_REPO)
tokenizer.push_to_hub(ADAPTER_REPO)
print("Done. Use this repo id in Phase 3 (app/model.py) to load the adapter.")
